# 03 - Customer Satisfaction and Verified-Purchase Analysis
## Objectives 1 and 2

**Ahsanullah University of Science and Technology** - Department of Computer Science and Engineering

**Course:** CSE 4262 Data Analytics Lab | **Lab Group:** Gr-03 | **Group:** Gr-06

| Student ID | Name |
|---|---|
| 20220104006 | A.S.M. Tahsin Tajware |
| 20220104014 | Abdullah Al Tamim |
| 20220104032 | Eusha Ahmed Mahi |


### Purpose

**Objective 1** quantifies satisfaction marketplace-wide and tests whether it holds across
categories as different as phones, games and beauty products.

**Objective 2** compares verified and non-verified reviews on rating, helpfulness and length. Any
gap is reported descriptively. A verified badge can still sit on a manipulated review when a
seller supplies a free unit, so the badge is measured here rather than trusted.

In [ ]:
import os, sys, glob

_candidates = ["/kaggle/working/repo", "/kaggle/working", "..", "."] + [
    os.path.dirname(p) for p in glob.glob("/kaggle/input/**/da_common.py", recursive=True)]
for _p in _candidates:
    if os.path.exists(os.path.join(_p, "da_common.py")):
        sys.path.insert(0, os.path.abspath(_p))
        break
else:
    raise FileNotFoundError("da_common.py not found. See KAGGLE_SETUP.md for the two setup options.")

from da_common import *

banner("Notebook 03 - Objectives 1 and 2")
spark = get_spark("03 satisfaction and verified")
df = load_analytical(spark).cache()
n_clean = df.count()
print(f"analytical dataset: {n_clean:,} rows")

### Objective 1 - Customer satisfaction

In [ ]:
overall = df.agg(F.round(F.avg("rating"), 3).alias("avg"), F.count("*").alias("n")).first()
print(f"Marketplace-wide average rating: {overall['avg']}  (n = {overall['n']:,})")

sat_overall = (df.groupBy("rating").count()
               .withColumn("pct", F.round(100 * F.col("count") / overall["n"], 1))
               .orderBy("rating"))
sat_overall.show()

sat_by_cat = (df.groupBy("Category")
              .agg(F.round(F.avg("rating"), 3).alias("avg_rating"),
                   F.round(100 * F.avg((F.col("rating") >= 4).cast("int")), 1).alias("pct_4_or_5"),
                   F.round(100 * F.avg((F.col("rating") <= 2).cast("int")), 1).alias("pct_1_or_2"),
                   F.count("*").alias("reviews"))
              .orderBy(F.desc("avg_rating")))
sat_by_cat.show(truncate=False)

save_table(sat_overall, "obj1_rating_distribution")
save_table(sat_by_cat, "obj1_satisfaction_by_category");

In [ ]:
cat_tot = df.groupBy("Category").count().withColumnRenamed("count", "cat_total")
cat_mix = (df.groupBy("Category", "rating").count()
           .join(cat_tot, "Category")
           .withColumn("share", 100 * F.col("count") / F.col("cat_total"))
           .groupBy("Category").pivot("rating", [1.0, 2.0, 3.0, 4.0, 5.0])
           .agg(F.round(F.first("share"), 1)))
print("Rating mix within each category (percent of that category)")
cat_mix.show()
save_table(cat_mix, "obj1_rating_mix_by_category");

In [ ]:
so = sat_overall.toPandas()
sbc = sat_by_cat.toPandas().sort_values("avg_rating")
pv = cat_mix.toPandas().set_index("Category")
pv.columns = [str(c) for c in pv.columns]
pv = pv.reindex([c for c in CATS if c in pv.index])

fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
ax[0].bar(so["rating"].astype(int).astype(str), so["pct"], color=STAR_COLORS)
ax[0].set_title("(a) Overall rating share"); ax[0].set_xlabel("Stars"); ax[0].set_ylabel("% of reviews")
for i, p in enumerate(so["pct"]):
    ax[0].text(i, p, f"{p}%", ha="center", va="bottom", fontsize=9)

ax[1].barh(sbc["Category"], sbc["avg_rating"], color=[ccol(c) for c in sbc["Category"]])
ax[1].set_xlim(max(0, sbc["avg_rating"].min() - 0.15), sbc["avg_rating"].max() + 0.15)
ax[1].set_title("(b) Average rating by category"); ax[1].set_xlabel("Average stars")
for i, v in enumerate(sbc["avg_rating"]):
    ax[1].text(v, i, f"  {v}", va="center", fontsize=10)

bottom = np.zeros(len(pv))
for star, col in zip(["1.0", "2.0", "3.0", "4.0", "5.0"], STAR_COLORS):
    if star not in pv.columns:
        continue
    vals = pv[star].astype(float).values
    ax[2].bar(pv.index, vals, bottom=bottom, label=f"{star[0]} star", color=col)
    bottom = bottom + vals
ax[2].set_title("(c) Rating mix within category"); ax[2].set_ylabel("% of category")
ax[2].legend(ncol=5, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, -0.30))

plt.tight_layout()
savefig(fig, "fig02_customer_satisfaction", "Objective 1 - satisfaction overall and by category")
plt.show()

### Objective 2 - Verified-purchase analysis

In [ ]:
verified_cmp = (df.groupBy("verified_purchase")
                .agg(F.count("*").alias("reviews"),
                     F.round(100 * F.count("*") / n_clean, 1).alias("share_pct"),
                     F.round(F.avg("rating"), 3).alias("avg_rating"),
                     F.round(F.avg("helpful_vote"), 3).alias("avg_helpful_vote"),
                     F.round(F.avg("review_length"), 1).alias("avg_review_length"),
                     F.round(100 * F.avg((F.col("rating") >= 4).cast("int")), 1).alias("pct_positive"))
                .orderBy(F.desc("verified_purchase")))
verified_cmp.show(truncate=False)

verified_by_cat = (df.groupBy("Category", "verified_purchase")
                   .agg(F.round(F.avg("rating"), 3).alias("avg_rating"),
                        F.round(F.avg("review_length"), 1).alias("avg_length"),
                        F.count("*").alias("reviews"))
                   .orderBy("Category", F.desc("verified_purchase")))
print("Verified vs non-verified within each category")
verified_by_cat.show(truncate=False)

save_table(verified_cmp, "obj2_verified_comparison")
save_table(verified_by_cat, "obj2_verified_by_category");

In [ ]:
vc = verified_cmp.toPandas()
vc["group"] = vc["verified_purchase"].map({True: "Verified", False: "Non-verified"})
palette = {"Verified": "#16a34a", "Non-verified": "#ef4444"}

fig, ax = plt.subplots(1, 4, figsize=(17, 4.3))
for a, (col, title) in zip(ax, [("avg_rating", "Average rating"),
                                ("avg_helpful_vote", "Average helpful votes"),
                                ("avg_review_length", "Average length (chars)"),
                                ("pct_positive", "% rated 4 or 5 stars")]):
    a.bar(vc["group"], vc[col], color=[palette[g] for g in vc["group"]])
    a.set_title(title); a.set_ylim(0, vc[col].max() * 1.25)
    for i, v in enumerate(vc[col]):
        a.text(i, v, f"{v}", ha="center", va="bottom", fontsize=10)

plt.suptitle("Objective 2 - verified vs non-verified reviews", y=1.04, fontsize=13, fontweight="bold")
plt.tight_layout()
savefig(fig, "fig03_verified_purchase", "Objective 2 - verified vs non-verified behaviour")
plt.show()

### Findings

Satisfaction is high and remarkably flat across categories, so category is a weak explanation of
rating on its own. The verified split is more informative than the rating gap alone suggests:
check the length and helpful-vote columns, where the two groups differ far more than they do on
stars. The per-category table is worth reading before drawing a conclusion, because the direction
of the rating gap is not guaranteed to be the same in every category.


In [ ]:
print(f"Tables in {TBL_DIR}")
for name in sorted(RESULTS):
    print(f"  {name:<40} {len(RESULTS[name]):>6,} rows")
print(f"\nFigures in {FIG_DIR}")
for name in sorted(FIGURES):
    print(f"  {name}.png")

In [ ]:
spark.stop()
print("Spark session stopped.")